
**This notebook is outdated. Last version here: [QLoRa: Fine-Tune a Large Language Model on Your GPU](https://open.substack.com/pub/kaitchup/p/qlora-fine-tune-a-large-language-model-on-your-gpu-27bed5a03e2b?r=2kp66c&utm_campaign=post&utm_medium=web)**

This notebook shows step-by-step how to fine-tune GPT-NeoX-20b with QLoRa. It works on a free instance of Google Colab.

For more details, you can read this post: [QLoRa: Fine-Tune a Large Language Model on Your GPU](https://open.substack.com/pub/kaitchup/p/qlora-fine-tune-a-large-language-model-on-your-gpu-27bed5a03e2b?r=2kp66c&utm_campaign=post&utm_medium=web)

First, we install all the dependencies.

In [1]:
!pip install -q -U bitsandbytes
!pip install -q -U transformers
!pip install -q -U peft
!pip install -q -U accelerate
!pip install -q -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 MB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


Import and load the tokenizer of GPT-NeoX-20B

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "EleutherAI/gpt-neox-20b"

#Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.08M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/457k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Configure the quantization with BitsAndBytes

In [4]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

Download, load, and quantize GPT NeoX on-the-fly.
I also enable gradient checkpointing to further reduce memory usage. You can skip it if you have enough VRAN on your GPU. It would significantly speed up training.



In [5]:
model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant_config, device_map={"":0})
model.gradient_checkpointing_enable()

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/60.4k [00:00<?, ?B/s]

model-00001-of-00046.safetensors:   0%|          | 0.00/926M [00:00<?, ?B/s]

model-00002-of-00046.safetensors:   0%|          | 0.00/910M [00:00<?, ?B/s]

model-00003-of-00046.safetensors:   0%|          | 0.00/910M [00:00<?, ?B/s]

...

model-00044-of-00046.safetensors:   0%|          | 0.00/910M [00:00<?, ?B/s]

model-00045-of-00046.safetensors:   0%|          | 0.00/604M [00:00<?, ?B/s]

model-00046-of-00046.safetensors:   0%|          | 0.00/620M [00:00<?, ?B/s]

The `GPTNeoXSdpaAttention` class is deprecated in favor of simply modifying the `config._attn_implementation`attribute of the `GPTNeoXAttention` class! It will be removed in v4.48


Loading checkpoint shards:   0%|          | 0/46 [00:00<?, ?it/s]

Configure low-rank adapters (LoRa). Note that if you want to fine-tune another LLM, you will likely have to change "target_modules" and "take_type".

In [6]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

Load a toy dataset for this demonstration.

In [7]:
from datasets import load_dataset

# Load the dataset
dataset_path = '/content/llm_traning_dataset.csv'  # Update with your dataset's path
data = load_dataset('csv', data_files=dataset_path)

# Display a sample of the dataset
print("Dataset Sample:")
print(data['train'][0])

# Preprocess the dataset
from transformers import AutoTokenizer

# Load the tokenizer for GPT-NeoX
tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neox-20b")

# Preprocess function to tokenize data
def preprocess_data(example):
    # Assuming your input column is named 'input' and output column is named 'output'
    # Adjust these names to match your actual column names in the CSV file.
    # Set the padding token to the EOS token before tokenizing
    tokenizer.pad_token = tokenizer.eos_token
    return {
        'input_ids': tokenizer(example['Prompt'], truncation=True, padding="max_length", max_length=512)['input_ids'],
        'labels': tokenizer(example['Response'], truncation=True, padding="max_length", max_length=512)['input_ids']
    }

# Apply preprocessing to the dataset
tokenized_data = data.map(preprocess_data, batched=True, remove_columns=['Prompt', 'Response']) # Remove original columns and use batched=True


Generating train split: 0 examples [00:00, ? examples/s]

Dataset Sample:
{'Prompt': 'Summarize the reviews for products in the \'Books & Literature\' category:\n- Review: "Purchased the Echo as a bridal shower gift. My niece and her fianc are still learning about all it can do but already reported back to me that they love it. I was told it is the most favorite gift they received." (Positive)\n- Review: "Initially I purchased this solely for the purchase of controlling Lutron lights. However, after being @ a friends house for a BBQ and watching them use this for music, I decided to give it a try to see if it could replace my Sonos.It can\'t, however, I\'ve found that I\'m much more likely to play Spotify through this using voice controls if I\'m in and out of a room for a shorter period of time and not near a Sonos to hit play/too busy to pull the phone out for Sonos app. This thing has also become more powerful as more IoT devices and services integrate with Echo or IFTTT. There is a TREMENDOUS amount of potential... give it a try and spend

Map:   0%|          | 0/65000 [00:00<?, ? examples/s]

Then, we can start training. I set up pad_token to eos_token since GPT NeoX doesn't have a pad token.
I put max_steps to 10 for demonstration, but you should fine-tune for at least 1 epoch (replace max_steps by num_train_epochs and change save_steps to 200 or more). It should take less than 3 hours for 1 epoch on the free instance of Google Colab (T4 GPU).

In [9]:
import transformers

tokenizer.pad_token = tokenizer.eos_token
model.config.use_cache = False
trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_data["train"], # Use tokenized_data here
    args=transformers.TrainingArguments(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=2,
        num_train_epochs=1,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=1,
        save_steps=10,
        output_dir="outputs",
        optim="paged_adamw_8bit",
        remove_unused_columns=False # Add this line
    ),
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
)
trainer.train()


/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,0.034800
2,0.033700
3,0.041100
4,0.029800
5,0.021100
6,0.067200
7,0.067000
8,0.065400
9,0.037400
10,0.054500


/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.10/dist-packages/torch/_dynamo/

KeyboardInterrupt: 

Testing inference:

In [20]:
text = "Summarise best three products from ipads category"
device = "cuda:0"
inputs = tokenizer(text, return_tensors="pt").to(device)
outputs = model.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Summarise best three products from ipads category

I am trying to get the best 3 products from the ipad category.
I have


In [21]:
prompt = """
Write a detailed blog article about the three best products in the "Electronics" category. Include:
1. The names of the top three products and their unique features.
2. Key differences between the products and when a customer should choose each.
3. Top complaints for each product.
"""

In [22]:
import torch

# Tokenize the input prompt
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to("cuda")

# Generate text using the model
output_ids = model.generate(
    input_ids,
    max_length=500,  # Maximum length of the generated text
    temperature=0.7, # Controls creativity (lower = more focused, higher = more creative)
    top_k=50,        # Limits sampling to the top-k tokens
    top_p=0.9,       # Nucleus sampling
    num_return_sequences=1  # Generate a single response
)

# Decode the output to human-readable text
output_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)

print("Generated Article:")
print(output_text)

/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/transformers/generation/configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected 

Generated Article:

Write a detailed blog article about the three best products in the "Electronics" category. Include:
1. The names of the top three products and their unique features.
2. Key differences between the products and when a customer should choose each.
3. Top complaints for each product.
4. How to find the best product for a customer.
5. How to find the best product for a customer.
6. How to find the best product for a customer.
7. How to find the best product for a customer.
8. How to find the best product for a customer.
9. How to find the best product for a customer.
10. How to find the best product for a customer.
11. How to find the best product for a customer.
12. How to find the best product for a customer.
13. How to find the best product for a customer.
14. How to find the best product for a customer.
15. How to find the best product for a customer.
16. How to find the best product for a customer.
17. How to find the best product for a customer.
18. How to find th